# 02 - Clean and encode

**Input:** `data/raw/match.csv`, `data/raw/set/<match>/set*.csv`, and the `shot_translations.py` module (Chinese -> English shot map).
**What it does:** walks every match folder, concatenates all set files into one long stroke table, translates shot types to English, computes opponent `displacement` for every stroke, and attaches the rally outcome (`hitter_won_rally`) to every stroke by forward/back-filling `getpoint_player` within each rally.
**Output:** `data/processed/strokes_all.csv` (one row per stroke, ~52k rows).

In [1]:
## --- path bootstrap: run from the repo root no matter where this is launched ---
## nbconvert and some editors set the working directory to the notebook's own
## folder. Walk up until we find the repo root (the folder containing data/),
## chdir there so relative data paths resolve, and put code/ on sys.path so the
## shared modules (utils.py, shot_translations.py) import cleanly.
import os, sys
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
os.chdir(_d)
if os.path.join(_d, "code") not in sys.path:
    sys.path.insert(0, os.path.join(_d, "code"))
print("working directory:", os.getcwd())

working directory: /Users/aakankshvaidya/Desktop/qss20_final_project


In [2]:
import os
import sys
import numpy as np
import pandas as pd

## import the shot-type translation dictionary from the code/ folder.
from shot_translations import SHOT_TYPE_MAP

In [3]:
## CONFIG -- relative paths only
RAW_DIR = "data/raw"
SET_DIR = os.path.join(RAW_DIR, "set")
PROC_DIR = "data/processed"
os.makedirs(PROC_DIR, exist_ok=True)

## Load match metadata and build a folder -> match lookup\nThe `video` column in match.csv equals the folder name under `set/`, so it links each stroke folder to its match row.

In [4]:
matches = pd.read_csv(os.path.join(RAW_DIR, "match.csv"))
matches_lookup = matches.set_index("video")[["id", "winner", "loser", "tournament", "round"]]
print("match rows:", len(matches))

match rows: 58


## Walk set folders and concatenate all strokes\nEach row is one stroke, tagged with its match identity.

In [5]:
set_folders = [f for f in os.listdir(SET_DIR) if os.path.isdir(os.path.join(SET_DIR, f))]

all_strokes = []
skipped_folders = []

for folder in set_folders:
    folder_path = os.path.join(SET_DIR, folder)
    ## skip folders not present in match.csv (defensive)
    if folder not in matches_lookup.index:
        skipped_folders.append(folder)
        continue
    match_info = matches_lookup.loc[folder]
    set_files = sorted([f for f in os.listdir(folder_path) if f.startswith("set") and f.endswith(".csv")])
    for sf in set_files:
        set_num = int(sf.replace("set", "").replace(".csv", ""))
        df = pd.read_csv(os.path.join(folder_path, sf))
        df["match_video"] = folder
        df["match_id"] = match_info["id"]
        df["match_winner"] = match_info["winner"]
        df["match_loser"] = match_info["loser"]
        df["tournament"] = match_info["tournament"]
        df["round"] = match_info["round"]
        df["set_num"] = set_num
        all_strokes.append(df)

strokes = pd.concat(all_strokes, ignore_index=True)
print("=== concatenated stroke data ===")
print("shape:", strokes.shape)
print("matches loaded:", strokes["match_id"].nunique())
print("skipped folders (not in match.csv):", len(skipped_folders))
if skipped_folders:
    print("  examples:", skipped_folders[:3])

=== concatenated stroke data ===
shape: (52356, 37)
matches loaded: 58
skipped folders (not in match.csv): 0


## Translate shot types to English

In [6]:
strokes["shot_type"] = strokes["type"].map(SHOT_TYPE_MAP)
unmapped = strokes[strokes["shot_type"].isna() & strokes["type"].notna()]["type"].unique()
print("unique raw shot types:", strokes["type"].nunique())
print("translated successfully:", strokes["shot_type"].notna().sum())
print("unmapped values:", list(unmapped) if len(unmapped) > 0 else "none")

unique raw shot types: 19
translated successfully: 52356
unmapped values: none


## Compute opponent displacement\nEuclidean distance from where the opponent stood to where the shuttle landed. Coordinates are dataset camera-pixel units, so this is a relative measure, not meters.

In [7]:
strokes["displacement"] = np.sqrt(
    (strokes["opponent_location_x"] - strokes["landing_x"]) ** 2
    + (strokes["opponent_location_y"] - strokes["landing_y"]) ** 2
)
print(strokes["displacement"].describe())

count    51472.000000
mean       155.955055
std         78.545659
min          1.000000
25%         98.412398
50%        147.383853
75%        202.408621
max        753.558226
Name: displacement, dtype: float64


## Attach rally outcome to every stroke\n`getpoint_player` is recorded only on the last stroke of each rally. Forward/back-fill it within each (match, set, rally) so every stroke knows who won the rally. This is a within-group fill, not a join, so the row count is unchanged.

In [8]:
n_before = len(strokes)
strokes["rally_winner"] = (
    strokes.groupby(["match_id", "set_num", "rally"])["getpoint_player"]
    .transform(lambda s: s.ffill().bfill())
)
strokes["rally_winner_is_match_winner"] = strokes["rally_winner"] == "A"
strokes["hitter_won_rally"] = strokes["player"] == strokes["rally_winner"]
print("rows before fill:", n_before, " after:", len(strokes))
print("strokes with a rally_winner:", strokes["rally_winner"].notna().sum(), "/", len(strokes))
print("share where hitter won the rally:", strokes["hitter_won_rally"].mean().round(3))

/var/folders/y1/dc8v042536gcqhm38hkl7c9c0000gn/T/ipykernel_96835/4258856289.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda s: s.ffill().bfill())


rows before fill: 52356  after: 52356
strokes with a rally_winner: 52105 / 52356
share where hitter won the rally: 0.491


## Save the processed stroke table

In [9]:
keep_cols = [
    "match_id", "match_video", "tournament", "round",
    "match_winner", "match_loser",
    "set_num", "rally", "ball_round",
    "player", "server", "type", "shot_type",
    "hit_area", "hit_x", "hit_y",
    "landing_area", "landing_x", "landing_y",
    "player_location_area", "player_location_x", "player_location_y",
    "opponent_location_area", "opponent_location_x", "opponent_location_y",
    "displacement",
    "lose_reason", "win_reason", "getpoint_player",
    "rally_winner", "hitter_won_rally",
]
keep_cols = [c for c in keep_cols if c in strokes.columns]
strokes_out = strokes[keep_cols].copy()
out_path = os.path.join(PROC_DIR, "strokes_all.csv")
strokes_out.to_csv(out_path, index=False)
print("rows:", len(strokes_out))
print("saved:", out_path)

rows: 52356
saved: data/processed/strokes_all.csv
